# Урок 10 - Агентные системы ИИ в продакшене

В этом уроке вы узнаете **паттерны продакшена** для агентов ИИ с использованием Microsoft Agent Framework (Python). Мы рассматриваем:

- **Наблюдаемость** — добавление измерения времени и логирования к взаимодействиям агента
- **Оценка** — использование агента-оценщика для оценки качества ответов
- **Управление затратами** — стратегии оптимизации токенов и выбора модели

Сценарий — **туристический агент**, который помогает пользователям планировать поездки, с мониторингом и оценкой, добавленными поверх.


## Настройка


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Производственные соображения

Перевод агентов ИИ из прототипов в производство требует тщательного внимания к трём столпам:

1. **Наблюдаемость** — Вам нужна видимость того, что делает агент, сколько времени это занимает и какие инструменты он вызывает. Без трассировки и логирования отладка проблем в рабочей среде практически невозможна.

2. **Оценка** — Автоматизированные проверки качества обеспечивают, что ответы агента остаются точными, полными и полезными с течением времени. Оценочный агент может присваивать ответам оценки по заданным критериям.

3. **Управление стоимостью** — Использование токенов напрямую влияет на стоимость. Стратегии, такие как оптимизация промптов, выбор модели и кэширование, помогают держать расходы под контролем без ущерба качеству.


## Создание наблюдаемого агента

Мы определяем инструменты путешествий и оборачиваем вызов агента с измерением времени, чтобы можно было отслеживать задержку. В продакшне вы бы интегрировали это с OpenTelemetry или аналогичным бэкендом трассировки.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Паттерны оценки

Распространённым шаблоном в продакшене является использование второго агента в качестве **оценщика**. Оценщик оценивает ответ основного агента по заранее определённым критериям, таким как полнота, точность и полезность.

This enables:
- Автоматические проверки качества до того, как ответы попадут к пользователям
- Обнаружение регрессий при изменении подсказок или моделей
- Непрерывный мониторинг работы агента с течением времени


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Стратегии управления затратами

Контроль затрат критически важен для производственных AI-агентов. Ниже приведены ключевые стратегии:

| Strategy | Description |
|---|---|
| **Оптимизация подсказок** | Держите системные инструкции краткими. Удаляйте избыточный контекст, чтобы сократить количество входных токенов. |
| **Выбор модели** | Используйте более компактные и дешёвые модели (например, GPT-4o-mini) для простых задач, таких как классификация или извлечение, а более крупные модели оставляйте для сложного рассуждения. |
| **Кэширование** | Кэшируйте результаты инструментов и частые запросы, чтобы избежать повторных вызовов API. |
| **Ограничения по токенам** | Установите ограничения `max_tokens`, чтобы предотвратить непредвиденно длинные ответы. |
| **Пакетирование** | Группируйте несколько пользовательских запросов в один вызов API, где это возможно. |

На практике хорошо работает многоуровневый подход: направляйте простые запросы на быструю и недорогую модель, а сложные — к более мощной.


## Резюме

В этом уроке вы узнали, как:

1. **Добавить наблюдаемость** в взаимодействия агента с помощью измерения времени и логирования, закладывая основу для трассировки и мониторинга.
2. **Оценивать ответы агента** автоматически с помощью агента-оценщика, который выставляет оценки за полноту, точность и полезность.
3. **Управлять затратами** за счёт оптимизации подсказок, выбора модели, кэширования и бюджетов токенов.

Эти производственные шаблоны помогают обеспечить надежность, измеримость и экономическую эффективность ваших агентов ИИ в масштабе.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведён с помощью сервиса перевода с использованием искусственного интеллекта [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматические переводы могут содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для критически важной информации рекомендуется воспользоваться услугами профессионального человеческого перевода. Мы не несем ответственности за любые недоразумения или неверные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
